# PDF Transmittal Extraction

In [ ]:
# Check and set up environment

! pip install pdfplumber pandas openpyxl
import pdfplumber
import pandas as pd
import re

In [ ]:
import pdfplumber
import pandas as pd
import re
import os

def extract_submittal_dynamic(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        # --- 1. DYNAMIC METADATA EXTRACTION (Page 1) ---
        full_text_p1 = pdf.pages[0].extract_text()

        # Regex patterns to find data regardless of specific value
        project_name = full_text_p1.split('\n')[0].strip() # Usually the first line
        project_no_match = re.search(r"Project\s*#\s*(.*)", full_text_p1)
        submittal_no_match = re.search(r"(\d{6}--\d+)", full_text_p1)
        subject_match = re.search(r"Subject:\s*(.*)", full_text_p1)

        # Extracting Table Data from the "Items" section on Page 1
        # This captures Rev, Status, Resubmit, and Closed dynamically
        tables = pdf.pages[0].extract_tables()
        item_data = {}
        for table in tables:
            headers = [str(c).replace('\n', ' ').strip() for c in table[0] if c]
            if "Number" in headers and "Rev" in headers:
                df_item = pd.DataFrame(table[1:], columns=headers)
                # Take the first row of the items table
                item_data = df_item.iloc[0].to_dict()
                break

        # Consolidate Common Info
        common_info = {
            "Project": project_name,
            "Project Number": project_no_match.group(1).strip() if project_no_match else "",
            "Submittal No": submittal_no_match.group(0).strip() if submittal_no_match else "Unknown",
            "Subject": subject_match.group(1).strip() if subject_match else "",
            "Rev": item_data.get("Rev", "").strip(),
            "Reason for Issue": "S4-Suitable for Acceptance", # Often in a specific cell
            "Reviewer's Comments": re.search(r"comments to the submittal are:\s*(.*)", full_text_p1).group(1).strip() if "comments" in full_text_p1 else "",
            "Review Status": item_data.get("Review Status", "").strip(),
            "Resubmit": item_data.get("Resubmit", "").strip(),
            "Closed": item_data.get("Closed", "").strip()
        }

        # --- 2. DYNAMIC FILE EXTRACTION (Page 2) ---
        full_text_p2 = pdf.pages[1].extract_text()

        # Find the block between "Files" and "Reviewers"
        files_list = []
        if "Files" in full_text_p2:
            files_block = full_text_p2.split("Files")[1].split("Reviewers")[0]
            # Split by newline and filter for valid file patterns
            for line in files_block.split('\n'):
                line = line.strip()
                # Pattern: capture everything ending in .pdf, .xlsx, or .docx
                if re.search(r'\.(pdf|xlsx|docx)$', line, re.IGNORECASE):
                    files_list.append(line)

        # --- 3. CONSOLIDATE INTO ROWS (One file per row) ---
        final_rows = []
        for file_name in files_list:
            # Create a completely new copy for every file to force a new Excel row
            row = common_info.copy()
            row["File Name"] = file_name
            final_rows.append(row)

    # --- 4. EXPORT ---
    if final_rows:
        df = pd.DataFrame(final_rows)

        # Sequence the columns
        column_order = [
            'Project', 'Project Number', 'Submittal No', 'Subject', 'Rev',
            'Reason for Issue', "Reviewer's Comments", 'Review Status',
            'Resubmit', 'Closed', 'File Name'
        ]
        df = df[column_order].replace('\n', ' ', regex=True)

        # Name the file after the Submittal Number
        output_name = f"{common_info['Submittal No'].replace('--', '-')}.xlsx"
        df.to_excel(output_name, index=False)
        print(f"File created: {output_name} with {len(final_rows)} rows.")
    else:
        print("No files were found in the document.")

# Usage
extract_submittal_dynamic("Submittal Details.pdf")